# 05. 지식베이스 기반 후보 코드 재정렬

04번 노트북의 단순 전체 문서 검색을 확장해, **SVM Top-3 후보 코드별로 관련 지식 문서를 연결**하고 후보별 근거 점수를 계산합니다. 포함 규칙과 제외 규칙은 분리하며, 제외 규칙을 해당 HS 코드의 긍정 근거로 사용하지 않습니다.

평가는 기존과 동일한 `StratifiedGroupKFold`를 사용합니다. 정답 HS 코드는 평가에만 사용하고, 후보 생성·검색·재정렬에는 사용하지 않습니다. 재정렬 가중치는 탐색적 설정이며, 성능 향상을 보장하지 않습니다.


In [ ]:
import json, re, requests, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.preprocessing import MinMaxScaler


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases = pd.read_csv(BASE + '완제품%20모델%20데이터_사례_전처리%20후%20-%20사례_전처리%20완료.csv').fillna('')
kb = pd.read_json(BASE + 'hs_knowledge_base.jsonl', lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
cases['model_text'] = cases[input_cols].astype(str).agg(' '.join, axis=1)
cases['duplicate_group'] = cases[input_cols].astype(str).agg('||'.join, axis=1).map(lambda x: __import__('hashlib').md5(x.encode()).hexdigest())
eligible = cases.groupby('HS Code').filter(lambda x: len(x) >= 4)
group_counts = eligible.groupby('HS Code')['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts >= 4].index.astype(str))
data = eligible[eligible['HS Code'].astype(str).isin(labels)].copy().reset_index(drop=True)
data['HS Code'] = data['HS Code'].astype(str)
print('cases:', len(data), '| classes:', data['HS Code'].nunique(), '| KB docs:', len(kb))
print(kb['source_sheet'].value_counts())


## 1. 코드 계층 및 규칙 유형 정규화

KB의 코드가 `3304.10`, `3304.10-9000`, `3304.00`처럼 서로 다른 자릿수로 기록될 수 있으므로 숫자 prefix 기반으로 계층 호환성을 판단합니다. 코드가 없는 일반 규정은 특정 후보 코드와 직접 일치한다고 간주하지 않습니다.


In [ ]:
def code_digits(value):
    return re.sub(r'\\D', '', str(value))

def code_compatible(rule_code, candidate_code):
    a, b = code_digits(rule_code), code_digits(candidate_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))

def is_exclusion(row):
    vals = ' '.join(str(row.get(c, '')) for c in ['source_sheet','rule_type','document_text'])
    return ('제외' in vals) or ('배제' in vals)

def is_positive_rule(row):
    vals = ' '.join(str(row.get(c, '')) for c in ['source_sheet','rule_type','document_text'])
    return (not is_exclusion(row)) and any(k in vals for k in ['포함','분류','소호별 포함 범위','정의·제조공정'])

kb['is_exclusion'] = kb.apply(is_exclusion, axis=1)
kb['is_positive'] = kb.apply(is_positive_rule, axis=1)
kb['code_digits'] = kb['hs_code'].map(code_digits)
print('positive:', int(kb['is_positive'].sum()), '| exclusion:', int(kb['is_exclusion'].sum()))


## 2. KB 검색기

문서 검색은 제품 입력정보와 규정 본문을 이용한 TF-IDF cosine similarity입니다. 후보 코드와의 계층 호환성은 검색 이후 별도 필터링합니다.


In [ ]:
kb['search_text'] = (kb['document_text'].astype(str) + ' ' + kb['rule_type'].astype(str) + ' ' + kb['source_sheet'].astype(str)).str.strip()
kb_vectorizer = FeatureUnion([('word', TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=30000, sublinear_tf=True)),('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), max_features=30000, sublinear_tf=True))])
kb_matrix = kb_vectorizer.fit_transform(kb['search_text'])
def search_kb(query, top_k=20):
    q = kb_vectorizer.transform([query])
    sims = (kb_matrix @ q.T).toarray().ravel()
    idx = np.argsort(sims)[::-1][:top_k]
    out = kb.iloc[idx].copy()
    out['similarity'] = sims[idx]
    return out


## 3. 후보 코드별 근거 점수

- **포함 근거 점수:** 검색 상위 문서 중 긍정적 분류 규칙이며, 후보 코드와 코드 계층이 호환되는 문서의 최대 유사도
- **제외 근거:** 별도 경고 목록으로 저장. 제외 규칙의 `hs_code`를 긍정 점수로 더하지 않음
- **일반 규정:** 코드가 특정되지 않은 공통 규정은 참고 근거로만 보존

기본 재정렬 점수는 SVM decision score에 작은 가중치의 포함 근거 점수를 더한 실험값입니다. 제외 규칙은 의미 해석이 필요한 경우가 많아 자동 감점하지 않습니다.


In [ ]:
def candidate_evidence(query, candidate, top_k=30):
    docs = search_kb(query, top_k=top_k)
    docs['code_match'] = docs['hs_code'].map(lambda x: code_compatible(x, candidate))
    positive = docs[docs['is_positive'] & docs['code_match']]
    exclusions = docs[docs['is_exclusion']]
    general = docs[(docs['code_digits'] == '') & (~docs['is_exclusion'])]
    pos_score = float(positive['similarity'].max()) if len(positive) else 0.0
    return pos_score, positive.head(3), exclusions.head(3), general.head(2)

def top_candidates(model, X, classes, n=3):
    scores = model.decision_function(X)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    order = np.argsort(scores, axis=1)[:, ::-1][:, :n]
    return [(classes[j], float(scores[i,j])) for i, row in enumerate(order) for j in row]


## 4. Group CV 및 OOF 비교

재정렬 계수 `ALPHA`는 우선 0.15로 고정합니다. 이 결과를 보고 가중치를 반복 조정하면 동일 CV 결과에 과적합될 수 있으므로, 최종 성능 주장은 별도 검증 설계가 필요합니다.


In [ ]:
N_SPLITS = 4
ALPHA = 0.15
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_rows, evidence_rows, fold_rows = [], [], []
for fold, (tr, te) in enumerate(cv.split(data['model_text'], data['HS Code'], groups=data['duplicate_group']), 1):
    train, test = data.iloc[tr], data.iloc[te]
    vec = FeatureUnion([('word', TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=1, max_features=30000, sublinear_tf=True)),('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=1, max_features=30000, sublinear_tf=True))])
    Xtr, Xte = vec.fit_transform(train['model_text']), vec.transform(test['model_text'])
    clf = LinearSVC(C=1, class_weight='balanced', max_iter=5000)
    clf.fit(Xtr, train['HS Code'])
    classes = clf.classes_
    decision = clf.decision_function(Xte)
    if decision.ndim == 1: decision = np.column_stack([-decision, decision])
    for local_i, (_, row) in enumerate(test.iterrows()):
        order = np.argsort(decision[local_i])[::-1][:3]
        candidates = [(str(classes[j]), float(decision[local_i,j])) for j in order]
        base_code = candidates[0][0]
        candidate_scores, candidate_details = [], []
        for code, svm_score in candidates:
            pos_score, pos_docs, excl_docs, general_docs = candidate_evidence(row['model_text'], code)
            rerank_score = svm_score + ALPHA * pos_score
            candidate_scores.append((code, svm_score, pos_score, rerank_score))
            candidate_details.append((code, pos_docs, excl_docs, general_docs))
            for kind, docs in [('positive_match',pos_docs),('exclusion_alert',excl_docs),('general_context',general_docs)]:
                for _, d in docs.iterrows():
                    evidence_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'candidate_hs':code,'evidence_kind':kind,'knowledge_id':d['knowledge_id'],'source_sheet':d['source_sheet'],'rule_hs_code':d['hs_code'],'rule_type':d['rule_type'],'similarity':float(d['similarity']),'document_text':d['document_text']})
        reranked = sorted(candidate_scores, key=lambda x:x[3], reverse=True)
        pred_rerank = reranked[0][0]
        oof_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'svm_top1':base_code,'svm_top3':' | '.join(x[0] for x in candidates),'reranked_top1':pred_rerank,'reranked_top3':' | '.join(x[0] for x in reranked),'svm_top1_score':candidates[0][1],'reranked_top1_score':reranked[0][3]})
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(train['duplicate_group']) & set(test['duplicate_group']))})
    print(f'fold {fold}: train={len(tr)}, test={len(te)}')

oof = pd.DataFrame(oof_rows)
evidence = pd.DataFrame(evidence_rows)
fold_audit = pd.DataFrame(fold_rows)
print('OOF:', len(oof), '| evidence:', len(evidence))
display(fold_audit)


## 5. 성능 및 근거 진단

Top-3 정확도는 SVM 원본 후보 목록 기준으로 계산합니다. 재정렬은 Top-3 후보 내부 순서만 바꾸므로 후보 집합 자체의 Top-3 포함 여부는 바뀌지 않습니다.


In [ ]:
def metric_row(name, pred):
    return {'method':name,'accuracy':accuracy_score(oof['true_hs'],pred),'macro_f1':f1_score(oof['true_hs'],pred,average='macro',zero_division=0),'weighted_f1':f1_score(oof['true_hs'],pred,average='weighted',zero_division=0)}

metrics = pd.DataFrame([metric_row('SVM baseline',oof['svm_top1']),metric_row('KB reranking',oof['reranked_top1'])])
oof['svm_top3_hit'] = oof.apply(lambda r: r['true_hs'] in r['svm_top3'].split(' | '),axis=1)
print('Top-3 accuracy:', oof['svm_top3_hit'].mean())
display(metrics)
print('Reranked changes:', int((oof['svm_top1'] != oof['reranked_top1']).sum()))
if len(evidence):
    display(evidence['evidence_kind'].value_counts().rename_axis('evidence_kind').to_frame('count'))
    display(evidence.groupby('evidence_kind')['similarity'].agg(['count','mean','max']))


## 6. 결과 저장

`kb_candidate_reranking_oof.csv`는 제품별 원본/재정렬 결과, `kb_candidate_reranking_evidence.csv`는 후보별 검색 근거, `kb_candidate_reranking_metrics.csv`는 비교 지표입니다.


In [ ]:
oof.to_csv('kb_candidate_reranking_oof.csv',index=False,encoding='utf-8-sig')
evidence.to_csv('kb_candidate_reranking_evidence.csv',index=False,encoding='utf-8-sig')
metrics.to_csv('kb_candidate_reranking_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('kb_candidate_reranking_fold_audit.csv',index=False,encoding='utf-8-sig')
print('저장 완료')
